In [2]:
import duckdb
con = duckdb.connect(r"C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb", read_only=True)

print("=" * 60)
print("CALIDAD DE LOS NUEVOS CAMPOS (con casteo)")
print("=" * 60)
stats = con.execute("""
    SELECT
        COUNT(*) AS total_filas,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS importe_pos,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) = 0 THEN 1 ELSE 0 END) AS importe_cero,
        SUM(CASE WHEN importe_total IS NULL OR importe_total = '' THEN 1 ELSE 0 END) AS importe_nulo,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) IS NULL AND importe_total IS NOT NULL AND importe_total != '' THEN 1 ELSE 0 END) AS no_castea,
        ROUND(AVG(TRY_CAST(importe_total AS DOUBLE)), 2) AS media,
        ROUND(MIN(TRY_CAST(importe_total AS DOUBLE)), 2) AS minimo,
        ROUND(MAX(TRY_CAST(importe_total AS DOUBLE)), 2) AS maximo
    FROM bronze.ventas_minoristas
""").fetchdf()
print(stats.to_string(index=False))

print("\nMuestra de 5 filas con importe > 0:")
muestra = con.execute("""
    SELECT importe_total, importe_total_con_descuento
    FROM bronze.ventas_minoristas
    WHERE TRY_CAST(importe_total AS DOUBLE) > 0
    LIMIT 5
""").fetchdf()
print(muestra.to_string(index=False))

print("\nValores raros (si hay alguno que no castea):")
raros = con.execute("""
    SELECT DISTINCT importe_total
    FROM bronze.ventas_minoristas
    WHERE TRY_CAST(importe_total AS DOUBLE) IS NULL
      AND importe_total IS NOT NULL 
      AND importe_total != ''
    LIMIT 10
""").fetchdf()
print(raros.to_string(index=False) if len(raros) > 0 else "(ninguno, todo castea limpio)")

con.close()

CALIDAD DE LOS NUEVOS CAMPOS (con casteo)
 total_filas  importe_pos  importe_cero  importe_nulo  no_castea  media  minimo  maximo
     2220352          0.0           0.0           0.0  2220352.0    NaN     NaN     NaN

Muestra de 5 filas con importe > 0:
Empty DataFrame
Columns: [importe_total, importe_total_con_descuento]
Index: []

Valores raros (si hay alguno que no castea):
importe_total
         null


In [3]:
import duckdb
con = duckdb.connect(r"C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb", read_only=True)

print("=" * 60)
print("VALORES DISTINTOS EN importe_total")
print("=" * 60)
distintos = con.execute("""
    SELECT 
        importe_total,
        COUNT(*) AS cuantas_filas
    FROM bronze.ventas_minoristas
    GROUP BY importe_total
    ORDER BY cuantas_filas DESC
    LIMIT 20
""").fetchdf()
print(distintos.to_string(index=False))

print("\n¿Cuántos valores distintos hay en total?")
n_distintos = con.execute("SELECT COUNT(DISTINCT importe_total) FROM bronze.ventas_minoristas").fetchone()[0]
print(f"   {n_distintos:,} valores distintos")

con.close()

VALORES DISTINTOS EN importe_total
importe_total  cuantas_filas
         null        2220352

¿Cuántos valores distintos hay en total?
   1 valores distintos


In [1]:
# Script de verificación rápida del nuevo ventas_minoristas.csv
# Ejecutar en una celda suelta de Jupyter o en una terminal con tu env activado

import pandas as pd
from pathlib import Path

RUTA = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark" / "data" / "raw" / "ventas_minoristas.csv"

print("="*70)
print("PASO 1 — Confirmar que el archivo existe y tamaño")
print("="*70)
print(f"Archivo: {RUTA}")
print(f"Existe : {RUTA.exists()}")
if RUTA.exists():
    tam_mb = RUTA.stat().st_size / (1024**2)
    print(f"Tamaño : {tam_mb:.1f} MB")

print("\n" + "="*70)
print("PASO 2 — Inspección de las primeras filas (sin cargar todo)")
print("="*70)
df_head = pd.read_csv(RUTA, nrows=5)
print("Columnas:", list(df_head.columns))
print(f"\nNúmero de columnas: {len(df_head.columns)}")
print(f"\nPrimeras 5 filas:\n{df_head.to_string()}")

print("\n" + "="*70)
print("PASO 3 — Verificar columnas críticas: importe_total e importe_total_con_descuento")
print("="*70)

# Leer SOLO las dos columnas críticas para ver si traen datos reales
columnas_check = ['importe_total', 'importe_total_con_descuento']
for col in columnas_check:
    if col not in df_head.columns:
        print(f"⚠️ Columna '{col}' NO existe en el CSV")
        continue
    # Leer una muestra de 100.000 filas para verificar el contenido
    df_check = pd.read_csv(RUTA, usecols=[col], nrows=100_000)
    n_null_textual = (df_check[col].astype(str) == 'null').sum()
    n_vacios = df_check[col].isna().sum()
    n_validos = len(df_check) - n_null_textual - n_vacios
    print(f"\nColumna '{col}' (muestra de 100.000 filas):")
    print(f"   · Valores numéricos válidos : {n_validos:>7} ({n_validos/len(df_check):.1%})")
    print(f"   · Texto 'null' literal      : {n_null_textual:>7} ({n_null_textual/len(df_check):.1%})")
    print(f"   · Vacíos / NaN              : {n_vacios:>7} ({n_vacios/len(df_check):.1%})")
    # Muestra de valores no nulos
    no_nulos = df_check[df_check[col].astype(str) != 'null'][col].dropna().head(5)
    if len(no_nulos) > 0:
        print(f"   · Ejemplos de valores       : {no_nulos.tolist()}")

print("\n" + "="*70)
print("PASO 4 — Conteo total de filas del CSV")
print("="*70)
n_filas = sum(1 for _ in open(RUTA, encoding='utf-8')) - 1
print(f"Total de filas: {n_filas:,}")
print(f"Esperado para 2019-2026: ~4 millones")
print(f"Esperado para 2022-2025: ~2,87 millones")

PASO 1 — Confirmar que el archivo existe y tamaño
Archivo: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\raw\ventas_minoristas.csv
Existe : True
Tamaño : 499.2 MB

PASO 2 — Inspección de las primeras filas (sin cargar todo)
Columnas: ['fecha_venta', 'cod_modelo', 'desc_modelo', 'cod_color', 'desc_color', 'cod_serie', 'desc_serie', 'talla', 'cantidad_neta', 'cantidad_ventas_sin_devoluciones', 'importe_total', 'importe_total_con_descuento', 'id_cliente', 'nombre_cliente', 'id_tipo_pedido', 'desc_tipo_pedido', 'id_temporada']

Número de columnas: 17

Primeras 5 filas:
  fecha_venta cod_modelo                   desc_modelo  cod_color desc_color  cod_serie desc_serie talla  cantidad_neta  cantidad_ventas_sin_devoluciones  importe_total  importe_total_con_descuento  id_cliente                      nombre_cliente  id_tipo_pedido desc_tipo_pedido  id_temporada
0  2019-09-19     10716B           SUJETADOR STRAPLESS         65     MARRON        107    GLACIER   110              1          

In [2]:
%pip install kaleido

  Using cached kaleido-1.3.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached choreographer-1.3.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached logistro-2.0.1-py3-none-any.whl.metadata (3.9 kB)
Using cached kaleido-1.3.0-py3-none-any.whl (55 kB)
Using cached choreographer-1.3.0-py3-none-any.whl (52 kB)
Using cached logistro-2.0.1-py3-none-any.whl (8.6 kB)

   ---------------------------------------- 0/5 [simplejson]
   ------------------------ --------------- 3/5 [choreographer]
   ------------------------ --------------- 3/5 [choreographer]
   ---------------------------------------- 5/5 [kaleido]

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Verificación de bibliotecas necesarias para los notebooks 09a y 09b
try:
    import plotly
    print(f"✅ plotly: {plotly.__version__}")
except ImportError:
    print("❌ plotly NO instalado")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    print(f"✅ plotly.express y plotly.graph_objects disponibles")
except ImportError:
    print("❌ submódulos plotly faltan")

try:
    import kaleido
    print(f"✅ kaleido: {kaleido.__version__}  (necesario para exportar a PNG)")
except ImportError:
    print("❌ kaleido NO instalado (haz: pip install kaleido)")

✅ plotly: 6.7.0
✅ plotly.express y plotly.graph_objects disponibles


AttributeError: module 'kaleido' has no attribute '__version__'

In [1]:
import duckdb
from pathlib import Path

RUTA_DUCKDB = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark" / "duckdb" / "selmark.duckdb"
con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

# Ver el esquema completo de silver.tiempo
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'tiempo'
    ORDER BY ordinal_position
""").fetchdf()
print("ESQUEMA REAL de silver.tiempo:")
print(esquema.to_string(index=False))

# Cuántos días tiene flag = TRUE en las columnas booleanas
print("\nDías con flag activo por columna booleana:")
bool_cols = esquema[esquema["data_type"] == "BOOLEAN"]["column_name"].tolist()
for col in bool_cols:
    n = con.execute(f"SELECT COUNT(*) FROM silver.tiempo WHERE {col} = TRUE").fetchone()[0]
    print(f"   {col:<30s}: {n:>4} días")

con.close()

ESQUEMA REAL de silver.tiempo:
        column_name data_type
              fecha      DATE
               anio   INTEGER
                mes   INTEGER
                dia   INTEGER
          trimestre   INTEGER
         semana_iso   INTEGER
       dia_del_anio   INTEGER
     dia_semana_num   INTEGER
         nombre_dia   VARCHAR
         nombre_mes   VARCHAR
           es_finde   BOOLEAN
       es_laborable   BOOLEAN
es_festivo_nacional   BOOLEAN
     nombre_festivo   VARCHAR
     temporada_moda   VARCHAR
         es_rebajas   BOOLEAN
    es_black_friday   BOOLEAN
    es_san_valentin   BOOLEAN
         es_navidad   BOOLEAN

Días con flag activo por columna booleana:
   es_finde                      :  418 días
   es_laborable                  : 1043 días
   es_festivo_nacional           :   35 días
   es_rebajas                    :  248 días
   es_black_friday               :    4 días
   es_san_valentin               :    4 días
   es_navidad                    :  124 días
